In [ ]:
# wavelength interpolation

import dLux as dl
import jax.numpy as np
import jax
import matplotlib.pyplot as plt
import dLux.utils as dlu
from dLux.sources import Source
from jax import Array
from dLux.sources import BaseSource
from jax import vmap
import jax.random as jr
from dLux import spectra
jax.config.update("jax_enable_x64", True)
plt.rcParams['image.origin'] = 'lower'

from jax.scipy.special import gammaln, xlogy

def poisson_loglike_gamma(k, mu):
    # analytic continuation using gammaln(k+1)
    k = np.asarray(k, dtype=float)
    mu = np.asarray(mu, dtype=float)
    valid = (k >= 0) & np.isfinite(k) & (mu > 0) & np.isfinite(mu)
    logp = xlogy(k, mu) - mu - gammaln(k + 1.0)
    return np.where(valid, logp, -np.inf)

# define the likelihood function:
def poiss_loglike(pytree, data) -> float:
    """
    Poissonian log likelihood of the pytree given the data. Assumes the pytree
    has a .model() function.

    Parameters
    ----------
    pytree : Base
        Pytree with a .model() function.
    data : Array
        Data to compare the model to.

    Returns
    -------
    log_likelihood : Array
        Log likelihood of the pytree given the data.
    """
    return poisson_loglike_gamma(pytree.model(), data).sum()

def normal_logpdf(x, mu, sigma):
    z = (x-mu)/sigma
    return -0.5 * (np.log(2*np.pi) + 2*np.log(sigma) + z**2)


### OPTICA FIGURES: INTEXT FONTSIZE 10. CAPTIONS GENERALLY 8

# JATIS style: 8 pt fonts, 5.25 in width, 0.5 pt spines and ticks (see jatis_style.py)
import jatis_style as js
js.apply()

fig_width = js.FIG_WIDTH


In [ ]:
eps = 1e-15
class CombinedSpectrum(dl.Spectrum):
    wavelengths: Array
    filt_weights: Array
    basis_weights: Array
    weights: Array
    flux: float

    def __init__(self, wavels, filt_weights, basis_weights):
        self.wavelengths = np.asarray(wavels, dtype=float)
        self.filt_weights = np.asarray(filt_weights, dtype=float)
        self.basis_weights = np.asarray(basis_weights, dtype=float)

        spec_w = self.spec_weights()
        weights = self.filt_weights * spec_w
        self.weights = weights / weights.sum()
        self.flux = np.sum(spec_w)

    def spec_weights(self):
        raise NotImplementedError

    def normalise(self):
        return self


class CombinedBasisSpectrum(CombinedSpectrum):
    basis_vects: Array

    def __init__(self, wavels, filt_weights, basis_weights, basis):
        self.basis_vects = np.asarray(basis, dtype=float)
        super().__init__(wavels, filt_weights, basis_weights)

    def spec_weights(self):
        return np.maximum(np.sum(self.basis_vects * self.basis_weights, axis=1), eps)
        # return 10 ** (np.sum(self.basis_vects * self.basis_weights, axis=1))

In [ ]:
wf_npixels = 128
diameter = 1.0
wf_oversample = 5

# changing
n_wavels_data = 101
psf_npixels = 20 # 10-100 lambda/D

coords = dlu.pixel_coords(wf_npixels*wf_oversample, diameter)

primary = dlu.circle(coords, diameter/2)

aperture = dlu.combine(np.array([primary]), wf_oversample)

aperture_layer = dl.TransmissiveLayer(aperture, True)

layers = [
    (
        'aperture',
        aperture_layer
    )
]


psf_pixel_scale = dlu.rad2arcsec((1.0e-6)/2) # nyquist
oversample = 2 # double oversample nyquist

optics = dl.AngularOpticalSystem(
    wf_npixels=wf_npixels,
    diameter=diameter,
    layers=layers,
    psf_npixels=psf_npixels,
    psf_pixel_scale=psf_pixel_scale,
    oversample=oversample
)


wlmin_data = 1.0e-6
wlmax_data = 2.0e-6

wavelengths_data = np.linspace((wlmin_data), (wlmax_data), n_wavels_data)

wlmin_data = np.min(wavelengths_data)
wlmax_data = np.max(wavelengths_data)

log_flux_data = 10 # billion photons?

nbasis = n_wavels_data

bandpass = np.ones(n_wavels_data)

spectrum_bins = CombinedBasisSpectrum(wavelengths_data, bandpass, np.ones(nbasis), np.eye(nbasis))

source_data = dl.PointSource(
    spectrum=spectrum_bins
)


model_data = dl.Telescope(optics, source_data)

def model_psf(spectrum, instrument):
    old_spec = instrument.get("source.spectrum")

    new_spec = CombinedBasisSpectrum(
        wavels=old_spec.wavelengths,
        filt_weights=old_spec.filt_weights,
        basis_weights=spectrum,
        basis=old_spec.basis_vects,
    )

    telescope = instrument.set("source.spectrum", new_spec)
    telescope = telescope.set("source.flux", telescope.get("source.spectrum").flux)
    return telescope.model()
    
injected_spectrum = np.ones(n_wavels_data)/n_wavels_data * 10**log_flux_data

# plt.plot(injected_spectrum)
img_full = model_psf(injected_spectrum, model_data)
#data = jr.poisson(jr.PRNGKey(0), img_full)

In [ ]:
from zodiax.diffops import hessian
import jax.scipy as jsp

def loss_fn(params, model, data):
    spectrum = params['spectrum']
    img = model_psf(spectrum, model)
    loss = -poisson_loglike_gamma(data, img).sum()
    return loss

params = {
    "spectrum": injected_spectrum
}

# let us do preconditioning with the data... this is fine because we are somewhat close to maximum likelihood
# since we can just do a flat spectrum with the same total flux

# upon later consideration.. don't do wrt. data because you get strange eigval trends

# upon even later consideration... DO do wrt. data. trust.
f = lambda params: loss_fn(params, model_data, img_full)

H, unflatten = hessian(f, params)

H = 0.5 * (H + H.T) # symmetretise

In [ ]:
# Now take appropriate eigvecs and vals

eigvals, eigvecs = np.linalg.eigh(H)

idx = np.argsort(eigvals)[::-1]

eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]
for i in range(eigvecs.shape[0]):
    eigvecs = eigvecs.at[:, i].mul(np.sign(np.mean(eigvecs[:1, i])))

mode_spectra = np.fft.rfft(eigvecs, axis=0)

In [ ]:
nmodes_plot_2 = 16
v = np.max(np.abs(eigvecs[:,:nmodes_plot_2]))

tick_idx = np.linspace(0, len(wavelengths_data)-1, 6, dtype=int)
freqs = np.fft.rfftfreq(n_wavels_data, d=(wavelengths_data[1]-wavelengths_data[0])*1e6)
freqtick_idx = np.linspace(0, len(freqs)-1, 6, dtype=int)

plt.figure(figsize=(fig_width,2*2.6125))

plt.subplot(2,1,2)
plt.imshow(eigvecs[:,:nmodes_plot_2].T, cmap='RdBu_r', aspect='auto', vmin=-v,vmax=v, interpolation='auto')
plt.xticks(
    tick_idx,
    [f'{wavelengths_data[i]*1e6:.1f}' for i in tick_idx],
    rotation=0
)
plt.xlabel(r'Wavelength $\bigl[\upmu\mathrm{m}\bigr]$')
plt.ylabel(r'Eigenmode Index')
cbar = plt.colorbar()
cbar.set_ticks([])
cbar.set_label(r'Amplitude')

ax = plt.gca()
ax.text(
    0.025, 0.125,
    '(b)',
    transform=ax.transAxes,
    ha='left',
    va='top',
    color='black',
)

plt.subplot(2,1,1)
plt.imshow((np.abs(mode_spectra[:,:nmodes_plot_2])).T, cmap='magma', aspect='auto', interpolation='auto',)
plt.xlabel(r'Frequency $\bigl[\upmu\mathrm{m}^{-1}\bigr]$')
plt.ylabel(r'Eigenmode Index')
plt.xticks(
    freqtick_idx,
    [f'{freqs[i]:.1f}' for i in freqtick_idx],
    rotation=0,
)
cbar = plt.colorbar()
cbar.set_ticks([])
cbar.set_label(r'Fourier Amplitude')

ax = plt.gca()
ax.text(
    0.025, 0.95,
    '(a)',
    transform=ax.transAxes,
    ha='left',
    va='top',
    color='white',
)

plt.tight_layout()
plt.subplots_adjust(right=1.05)

# plt.savefig('goodfig16modes1ab.pdf', dpi=300) #bbox_inches='tight')
plt.show()

In [ ]:
v = np.max(np.abs(eigvecs[:,:nmodes_plot_2]))

tick_idx = np.linspace(0, len(wavelengths_data)-1, 6, dtype=int)
freqs = np.fft.rfftfreq(n_wavels_data, d=(wavelengths_data[1]-wavelengths_data[0])*1e6)
freqtick_idx = np.linspace(0, len(freqs)-1, 6, dtype=int)
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'

plt.figure(figsize=(fig_width,fig_width*0.4))

plt.subplot(1,2,1)
plt.imshow(eigvecs[:,:nmodes_plot_2].T, cmap='RdBu_r', aspect='auto', vmin=-v,vmax=v, interpolation='auto')
plt.xticks(
    tick_idx,
    [f'{wavelengths_data[i]*1e6:.1f}' for i in tick_idx],
    rotation=0
)
plt.xlabel(r'Wavelength $\bigl[\upmu\mathrm{m}\bigr]$')
plt.ylabel(r'Eigenmode index')
cbar = plt.colorbar()
cbar.set_ticks([0])
cbar.set_ticklabels([])
cbar.set_label(r'Amplitude')

ax = plt.gca()
ax.text(
    0.08, 0.92,
    '(a)',
    transform=ax.transAxes,
    ha='left',
    va='top',
    color='black',
)

plt.subplot(1,2,2)
plt.imshow((np.abs(mode_spectra[:,:nmodes_plot_2])).T, cmap='viridis', aspect='auto', interpolation='auto')
plt.xlabel(r'Frequency $\bigl[\upmu\mathrm{m}^{-1}\bigr]$')
# plt.ylabel('Eigenmode Index')
plt.yticks(ticks=[0, 5, 10 ,15 ], labels=[])
plt.xticks(
    freqtick_idx,
    [f'{freqs[i]:.1f}' for i in freqtick_idx],
    rotation=0,
)
cbar = plt.colorbar()
cbar.set_ticks([np.max((np.abs(mode_spectra[:,:nmodes_plot_2])).T)/2])
cbar.set_ticklabels([
])
cbar.set_label(r'Fourier amplitude')

ax = plt.gca()
ax.text(
    0.08, 0.92,
    '(b)',
    transform=ax.transAxes,
    ha='left',
    va='top',
    color='white',
)

# plt.tight_layout()

plt.subplots_adjust(right=1.01)
# plt.savefig('goodfig16modes2abin.pdf', dpi=300,) #bbox_inches='tight')
plt.show()

In [ ]:
v = np.max(np.abs(eigvecs[:, :nmodes_plot_2]))

tick_idx = np.linspace(0, len(wavelengths_data) - 1, 6, dtype=int)
freqs = np.fft.rfftfreq(
    n_wavels_data,
    d=(wavelengths_data[1] - wavelengths_data[0]) * 1e6,
)
freqtick_idx = np.linspace(0, len(freqs) - 1, 6, dtype=int)

plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"

fig, axes = plt.subplots(
    1, 4,
    figsize=(fig_width, fig_width * 0.4),
    gridspec_kw={"width_ratios": [1, 0.035, 1, 0.035]},
    layout="constrained",
)
ax1, cax1, ax2, cax2 = axes

fig.get_layout_engine().set(
    w_pad=0.025,
    h_pad=0.025,
    wspace=0.02,
)

# Eigenmodes
im1 = ax1.imshow(
    eigvecs[:, :nmodes_plot_2].T,
    cmap="RdBu_r",
    aspect="auto",
    vmin=-v,
    vmax=v,
    interpolation="auto",
)

ax1.set_xticks(
    tick_idx,
    labels=[f"{wavelengths_data[i] * 1e6:.1f}" for i in tick_idx],
)
ax1.set_xlabel(r"Wavelength $\bigl[\upmu\mathrm{m}\bigr]$")
ax1.set_ylabel("Eigenmode index")
ax1.set_yticks([0, 5, 10, 15])

cbar1 = fig.colorbar(im1, cax=cax1)
cbar1.set_ticks([0], labels=[""])
cbar1.set_label("Amplitude")

js.panel_label(ax1, "a", x=0.075, y=0.92, color="black", bbox=js.LABEL_BBOX)

# Fourier amplitudes
fourier_amplitude = np.abs(mode_spectra[:, :nmodes_plot_2]).T

im2 = ax2.imshow(
    fourier_amplitude,
    cmap="viridis",
    aspect="auto",
    interpolation="auto",
)

ax2.set_xlabel(r"Frequency $\bigl[\upmu\mathrm{m}^{-1}\bigr]$")
ax2.set_yticks([0, 5, 10, 15], labels=["", "", "", ""])
ax2.set_xticks(
    freqtick_idx,
    labels=[f"{freqs[i]:.1f}" for i in freqtick_idx],
)

cbar2 = fig.colorbar(im2, cax=cax2)
cbar2.set_ticks([np.max(fourier_amplitude) / 2], labels=[""])
cbar2.set_label("Fourier amplitude")

js.panel_label(ax2, "b", x=0.075, y=0.92, color="white")

fig.savefig("goodfig_jatis.pdf", dpi=300, bbox_inches=None, backend='pgf')
plt.show()

In [ ]:
### Now we have to do a figure showing spectrum upto x number of modes
# And convert the uncertainties in coefficient space to uncertainties in wavelength space
a = eigvecs.T @ injected_spectrum
relstd = 1/(np.abs(a[0]) * np.sqrt(eigvals))
print(np.argmax(relstd>0.1))
plt.plot((1/(np.abs(a[0]) * np.sqrt(eigvals)))[:10], '.-')

In [ ]:
relstd>0.1

In [ ]:
nmodes_plot = 9
spec_mean = eigvecs[:,:nmodes_plot] @ a[:nmodes_plot]
spec_var = (eigvecs[:,:nmodes_plot]**2) @ (1/(eigvals[:nmodes_plot]))
spec_std = np.sqrt(spec_var)

plt.plot(
    wavelengths_data*1e6,
    spec_mean,
    color='green',
    label=f'recovered spectrum, nmodes={nmodes_plot}'
)
plt.fill_between(
    wavelengths_data*1e6,
    spec_mean - spec_std,
    spec_mean + spec_std,
    color='green',
    alpha=0.3
)
plt.ylim(0, 1.2*np.max(injected_spectrum))

In [ ]:
a = eigvecs.T @ injected_spectrum

nmodes_plot = 4

V = eigvecs[:, :nmodes_plot]
spec_mean = V @ a[:nmodes_plot]
spec_var = (V**2) @ (1 / eigvals[:nmodes_plot])
spec_std = np.sqrt(spec_var)

x = wavelengths_data * 1e6

plt.figure(figsize=(fig_width, 0.5*fig_width))
plt.errorbar(
    x,
    spec_mean,
    yerr=spec_std,
    fmt='x',
    markersize=4,
    capsize=2,
    label=fr'recovered spectrum, $N_\mathrm{{modes}}={nmodes_plot}$'
)

plt.plot(
    x,
    injected_spectrum,
    'k--',
    linewidth=1,
    label=r'True Spectrum'
)

# plt.ylim(0, 1.2 * np.max(injected_spectrum))
plt.xlabel(r'Wavelength $\bigl[\upmu \mathrm{m} \bigr]$')
plt.ylabel('Photon flux per wavelength bin')
plt.legend()

In [ ]:
# I think this figure might not be necessary, and could be misleading in the flat spectrum case
# Much more suited toward injection/recovery.
# Instead let us derive the FOV/spectral modes.
# And total flux/SNR spectral modes relationships.
# Like a grid search, can just do a 2D figure.

In [ ]:
# plotting the eigenmodes

spectrum_2 = CombinedBasisSpectrum(wavelengths_data, bandpass, np.zeros(nmodes_plot_2), eigvecs[:,:nmodes_plot_2])

source_2 = dl.PointSource(
    spectrum=spectrum_2
)

wf_npixels = 128
diameter = 1.0
wf_oversample = 5

# changing
n_wavels_data = 101
psf_npixels = 20 # 10-100 lambda/D

coords = dlu.pixel_coords(wf_npixels*wf_oversample, diameter)

primary = dlu.circle(coords, diameter/2)

aperture = dlu.combine(np.array([primary]), wf_oversample)

aperture_layer = dl.TransmissiveLayer(aperture, True)

layers = [
    (
        'aperture',
        aperture_layer
    )
]


psf_pixel_scale = dlu.rad2arcsec((1.0e-6)/2) # nyquist
oversample = 16 # double oversample nyquist

optics = dl.AngularOpticalSystem(
    wf_npixels=wf_npixels,
    diameter=diameter,
    layers=layers,
    psf_npixels=psf_npixels,
    psf_pixel_scale=psf_pixel_scale,
    oversample=oversample
)

model_2 = dl.Telescope(optics,source_2)

# diff1 = model_psf(np.array([1e6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]), model_2) - model_psf(np.array([1e6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]), model_2)
# diff1 = np.abs(diff1)**0.5 * np.sign(diff1)
# v = np.max(np.abs(diff1))

plt.figure(figsize=(fig_width,fig_width))
for i in range(16):

    params = np.zeros(16)
    params = params.at[0].set(1e6)
    params = params.at[i].add(1)

    diff1 = model_psf(params, model_2) - model_psf(
        np.array([1e6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]), model_2
    )

    diff1 = np.abs(diff1)**0.5 * np.sign(diff1)
    v = np.max(np.abs(diff1))

    plt.subplot(4,4,i+1)
    plt.imshow(diff1, cmap='berlin_r', vmin=-v, vmax=v)
    # plt.colorbar()
    plt.text(
        s=f'({i})', 
        color='white',
        x=10,
        y=oversample*psf_npixels-10,
        ha='left',
        va='top',
        )
    plt.xticks([])
    plt.yticks([])

plt.show()

In [ ]:
fig = plt.figure(figsize=(fig_width, fig_width))

for i in range(16):

    params = np.zeros(16)
    params = params.at[0].set(1e6)
    params = params.at[i].add(1)

    diff1 = model_psf(params, model_2) - model_psf(
        np.array([1e6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]), model_2
    )

    diff1 = np.abs(diff1)**0.5 * np.sign(diff1)
    v = np.max(np.abs(diff1))

    ax = plt.subplot(4, 4, i + 1)
    ax.imshow(diff1, cmap='berlin_r', vmin=-v, vmax=v)

    ax.text(
        s=f'({i})',
        color='white',
        x=10,
        y=oversample * psf_npixels - 10,
        ha='left',
        va='top',
    )

    ax.set_xticks([])
    ax.set_yticks([])

# Leave room on the right for one illustrative colourbar
fig.subplots_adjust(right=0.90)

# Skinny colourbar axis: [left, bottom, width, height]
cax = fig.add_axes([0.92, 0.12, 0.02, 0.74])

# Illustrative colourbar only — independent of the individual panel scales
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

sm = ScalarMappable(
    norm=Normalize(vmin=-1, vmax=1),
    cmap='berlin_r'
)

cbar = fig.colorbar(sm, cax=cax)

# Tick positions corresponding to a signed square-root scaling
fractions = np.array([
    -1, -0.75, -0.5, -0.25,
     0,
     0.25, 0.5, 0.75, 1
])

ticks = np.sign(fractions) * np.sqrt(np.abs(fractions))

cbar.set_ticks(ticks)

# Show tick marks but no numerical labels
cbar.set_ticklabels([''] * len(ticks))
cbar.set_label('test', labelpad=2)

plt.show()

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

fig = plt.figure(figsize=(fig_width, fig_width))
axs = []

for i in range(16):

    params = np.zeros(16)
    params = params.at[0].set(1e6)
    params = params.at[i].add(1)

    diff1 = model_psf(params, model_2) - model_psf(
        np.array([1e6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]), model_2
    )

    diff1 = np.abs(diff1)**0.5 * np.sign(diff1)
    v = np.max(np.abs(diff1))

    ax = plt.subplot(4, 4, i + 1)
    axs.append(ax)

    ax.imshow(diff1, cmap='berlin_r', vmin=-v, vmax=v)

    ax.text(
        s=f'({i})',
        color='white',
        x=10,
        y=oversample * psf_npixels - 10,
        ha='left',
        va='top',
    )

    ax.set_xticks([])
    ax.set_yticks([])

# leave room on the right
fig.subplots_adjust(right=0.90)

# get exact top/bottom/right extent of the subplot grid
top = max(ax.get_position().y1 for ax in axs)
bottom = min(ax.get_position().y0 for ax in axs)
right = max(ax.get_position().x1 for ax in axs)

# create a skinny colorbar axis aligned with the image grid
pad = 0.02
cb_width = 0.02
cax = fig.add_axes([right + pad, bottom, cb_width, top - bottom])

# illustrative colourbar only
sm = ScalarMappable(
    norm=Normalize(vmin=-1, vmax=1),
    cmap='berlin_r'
)

cbar = fig.colorbar(sm, cax=cax)

# signed sqrt tick positions
fractions = np.array([-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1])
ticks = np.sign(fractions) * np.sqrt(np.abs(fractions))

cbar.set_ticks(ticks)
cbar.set_ticklabels([''] * len(ticks))

# label closer to the bar
cbar.set_label('Eigenmode Jacobian Amplitude', labelpad=2)

plt.show()

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

fig = plt.figure(figsize=(fig_width, fig_width))

# Alternating image row / eigenmode row
gs = fig.add_gridspec(
    8, 4,
    height_ratios=[4, 1, 4, 1, 4, 1, 4, 1],
    hspace=0.05,
    wspace=0.05
)

image_axes = []
mode_axes = []

for i in range(16):

    row = i // 4
    col = i % 4

    # --------------------
    # PSF/Jacobian image
    # --------------------
    ax = fig.add_subplot(gs[2*row, col])
    image_axes.append(ax)

    params = np.zeros(16)
    params = params.at[0].set(1e6)
    params = params.at[i].add(1)

    diff1 = model_psf(params, model_2) - model_psf(
        np.array([1e6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]), model_2
    )

    diff1 = np.abs(diff1)**0.5 * np.sign(diff1)
    v = np.max(np.abs(diff1))

    ax.imshow(diff1, cmap='berlin_r', vmin=-v, vmax=v)

    ax.text(
        10,
        oversample * psf_npixels - 10,
        f'({i})',
        color='white',
        ha='left',
        va='top',
    )

    ax.set_xticks([])
    ax.set_yticks([])

    # --------------------
    # Spectral eigenmode
    # --------------------
    ax_mode = fig.add_subplot(gs[2*row + 1, col])
    mode_axes.append(ax_mode)
    
    ax_mode.plot(wavelengths_data, eigvecs[:, i])
    ax_mode.axhline(0, lw=0.5, alpha=0.5)

    # Keep these minimal so they don't clutter the figure
    ax_mode.set_yticks([])

    # Only show wavelength labels on bottom row
    if row < 3:
        ax_mode.set_xticklabels([])
    else:
        ax_mode.set_xlabel(r'Wavelength')

# leave room for colourbar
fig.subplots_adjust(right=0.90)

# Get exact vertical extent of EVERYTHING
top = max(ax.get_position().y1 for ax in image_axes)
bottom = min(ax.get_position().y0 for ax in mode_axes)
right = max(ax.get_position().x1 for ax in image_axes)

# Illustrative colourbar
pad = 0.02
cb_width = 0.015

cax = fig.add_axes([
    right + pad,
    bottom,
    cb_width,
    top - bottom
])

sm = ScalarMappable(
    norm=Normalize(vmin=-1, vmax=1),
    cmap='berlin_r'
)

cbar = fig.colorbar(sm, cax=cax)

fractions = np.array([
    -1, -0.75, -0.5, -0.25,
     0,
     0.25, 0.5, 0.75, 1
])

ticks = np.sign(fractions) * np.sqrt(np.abs(fractions))

cbar.set_ticks(ticks)
cbar.set_ticklabels([''] * len(ticks))

cbar.set_label('Eigenmode Jacobian', labelpad=2)

plt.show()

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import matplotlib.patheffects as pe

# path_effects = [
#     pe.Stroke(linewidth=1.5, foreground="k", alpha=1.0),
#     pe.Normal(),
# ]

fig = plt.figure(figsize=(fig_width, 1.15*fig_width))

# First make only the image grid
gs = fig.add_gridspec(
    4, 4,
    left=0.02, right=0.92, bottom=0.04, top=1.00,
    wspace=-0.0, hspace=0.00
)

image_axes = []
ims = []

for i in range(16):
    row = i // 4
    col = i % 4

    ax = fig.add_subplot(gs[row, col])
    image_axes.append(ax)

    params = np.zeros(16)
    params = params.at[0].set(1e6)
    params = params.at[i].add(1)

    diff1 = model_psf(params, model_2) - model_psf(
        np.array([1e6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]), model_2
    )

    diff1 = np.abs(diff1)**0.5 * np.sign(diff1)
    v = np.max(np.abs(diff1))

    im = ax.imshow(diff1, cmap='berlin_r', vmin=-v, vmax=v)
    ims.append(im)

    ax.text(
        10,
        oversample * psf_npixels - 10,
        f'({i})',
        color='white',
        ha='left',
        va='top',
    )

    ax.set_xticks([])
    ax.set_yticks([])

# Important: draw once so positions are final
fig.canvas.draw()

# Now add mode axes manually underneath each image axis,
# matching the exact displayed width of the image axes
mode_axes = []
mode_height = 0.045   # height in figure coordinates
mode_gap = 0 # 0.012      # vertical gap below each image

ev = np.max(np.abs(eigvecs[:, :nmodes_plot_2]))

for i, ax_img in enumerate(image_axes):
    pos = ax_img.get_position()

    ax_mode = fig.add_axes([
        pos.x0,                  # same left edge
        pos.y0 - mode_gap - mode_height,  # directly underneath
        pos.width,               # same width
        mode_height
    ])
    mode_axes.append(ax_mode)

    ax_mode.set_facecolor('black')
    ax_mode.axhline(0, alpha=0.5, color='0.75', ls='--', linewidth=1,)#  path_effects=path_effects)
    ax_mode.plot(wavelengths_data, eigvecs[:, i], color='white', linewidth=1)

    ax_mode.set_xlim(wavelengths_data[0], wavelengths_data[-1])
    ax_mode.set_ylim(-ev, ev)
    ax_mode.set_yticks([])
    ax_mode.set_xticks([])

    # only bottom row gets x tick labels
    if i // 4 < 3:
        ax_mode.set_xticklabels([])
    else:
        # ax_mode.set_xlabel('Wavelength')
        ax_mode.set_xticklabels([])
        # optional: fewer ticks if cluttered
        # ax_mode.set_xticks([wavelengths[0], wavelengths[len(wavelengths)//2], wavelengths[-1]])

# ---- colourbar aligned to image grid only ----
top = max(ax.get_position().y1 for ax in image_axes)
bottom = min(ax_mode.get_position().y0 for ax in image_axes)
right = max(ax.get_position().x1 for ax in image_axes)

pad = 0.02
cb_width = 0.02
cax = fig.add_axes([right + pad, bottom, cb_width, top - bottom])

sm = ScalarMappable(
    norm=Normalize(vmin=-1, vmax=1),
    cmap='berlin_r'
)

cbar = fig.colorbar(sm, cax=cax)

fractions = np.array([-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1])
ticks = np.sign(fractions) * np.sqrt(np.abs(fractions))

cbar.set_ticks(ticks)
cbar.set_ticklabels([''] * len(ticks))
cbar.set_label('Eigenmode Jacobian Amplitude', labelpad=2)

for ax in fig.axes:
    for spine in ax.spines.values():
        spine.set_color('grey')

    ax.tick_params(
        axis='both',
        colors='grey'
    )
        
plt.savefig('eigenmode_jacobian_figure_4.pdf', dpi=300)
plt.show()

In [ ]:
plt.imshow(eigvecs[:,:nmodes_plot_2])

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import matplotlib.patheffects as pe

# path_effects = [
#     pe.Stroke(linewidth=1.5, foreground="k", alpha=1.0),
#     pe.Normal(),
# ]

# First make only the image grid
fig = plt.figure(figsize=(fig_width, 0.575 * fig_width))

gs = fig.add_gridspec(
    2, 4,
    left=0.02, right=0.92, bottom=0.08, top=1.00,
    wspace=0.0, hspace=0.00,
)

image_axes = []
ims = []

for i in range(8):
    row = i // 4
    col = i % 4

    ax = fig.add_subplot(gs[row, col])
    image_axes.append(ax)

    params = np.zeros(16)
    params = params.at[0].set(1e6)
    params = params.at[i].add(1)

    diff1 = model_psf(params, model_2) - model_psf(
        np.array([1e6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]), model_2
    )

    diff1 = np.abs(diff1)**0.5 * np.sign(diff1)
    v = np.max(np.abs(diff1))

    im = ax.imshow(diff1, cmap='berlin_r', vmin=-v, vmax=v)
    ims.append(im)

    ax.text(
        10,
        oversample * psf_npixels - 10,
        f'({i})',
        color='white',
        ha='left',
        va='top',
    )

    ax.set_xticks([])
    ax.set_yticks([])

# Important: draw once so positions are final
fig.canvas.draw()

# Now add mode axes manually underneath each image axis,
# matching the exact displayed width of the image axes
mode_axes = []
mode_height = 0.069 #8   # height in figure coordinates
mode_gap = 0 # 0.012      # vertical gap below each image

ev = np.max(np.abs(eigvecs[:, :nmodes_plot_2]))

for i, ax_img in enumerate(image_axes):
    pos = ax_img.get_position()

    ax_mode = fig.add_axes([
        pos.x0,                  # same left edge
        pos.y0 - mode_gap - mode_height,  # directly underneath
        pos.width,               # same width
        mode_height
    ])
    mode_axes.append(ax_mode)

    ax_mode.set_facecolor('black')
    ax_mode.axhline(0, alpha=0.5, color='0.75', ls='--', linewidth=0.8,)#  path_effects=path_effects)
    ax_mode.plot(wavelengths_data, eigvecs[:, i], color='white', linewidth=0.8)

    ax_mode.set_xlim(wavelengths_data[0], wavelengths_data[-1])
    ax_mode.set_ylim(-ev, ev)
    ax_mode.set_yticks([])
    ax_mode.set_xticks([])

    # only bottom row gets x tick labels
    if i // 4 < 3:
        ax_mode.set_xticklabels([])
    else:
        # ax_mode.set_xlabel('Wavelength')
        ax_mode.set_xticklabels([])
        # optional: fewer ticks if cluttered
        # ax_mode.set_xticks([wavelengths[0], wavelengths[len(wavelengths)//2], wavelengths[-1]])

# ---- colourbar aligned to image grid only ----
top = max(ax.get_position().y1 for ax in image_axes)
bottom = min(ax.get_position().y0 for ax in mode_axes)#min(ax_mode.get_position().y0 for ax in image_axes)
right = max(ax.get_position().x1 for ax in image_axes)

pad = 0.02
cb_width = 0.02
cax = fig.add_axes([right + pad, bottom, cb_width, top - bottom])

sm = ScalarMappable(
    norm=Normalize(vmin=-1, vmax=1),
    cmap='berlin_r'
)

cbar = fig.colorbar(sm, cax=cax)

fractions = np.array([-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1])
ticks = np.sign(fractions) * np.sqrt(np.abs(fractions))

cbar.set_ticks(ticks)
cbar.set_ticklabels([''] * len(ticks))
cbar.set_label('Eigenmode Jacobian amplitude', labelpad=2)

for ax in fig.axes:
    for spine in ax.spines.values():
        spine.set_color('grey')

    ax.tick_params(
        axis='both',
        colors='grey'
    )
        
plt.savefig('eigenmode_jacobian_jatis.pdf', dpi=300)
plt.show()